In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler, SQLTransformer

# 1. INITIALIZE SPARK
spark = SparkSession.builder \
    .appName("NYC_Taxi_Stage6_Pipeline") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("Loading Train, Validation, and Test splits...")
train_df = spark.read.parquet("../data/processed/train.parquet")
val_df = spark.read.parquet("../data/processed/val.parquet")
test_df = spark.read.parquet("../data/processed/test.parquet")

# 2. FEATURE ENGINEERING: SQLTransformer
# Extracting the hour from datetime (Domain Knowledge from Stage 4 - Query 2)
sql_trans = SQLTransformer(
    statement="SELECT *, HOUR(CAST(tpep_pickup_datetime AS TIMESTAMP)) AS pickup_hour FROM __THIS__"
)

# 3. CATEGORICAL ENCODING
cat_cols = ["RateCodeID", "VendorID", "pickup_hour"]
# handleInvalid="keep" ensures the model doesn't crash if it sees a new category in Test
indexers = [StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep") for c in cat_cols]
encoders = [OneHotEncoder(inputCol=c+"_idx", outputCol=c+"_vec") for c in cat_cols]

# 4. NUMERICAL SCALING
# We explicitly exclude 'total_amount' and 'tip_amount' to prevent Data Leakage
num_cols = ["passenger_count", "trip_distance", "fare_amount", "extra", "mta_tax", "tolls_amount", "improvement_surcharge"]

num_assembler = VectorAssembler(inputCols=num_cols, outputCol="num_features")
scaler = StandardScaler(inputCol="num_features", outputCol="scaled_num_features", withStd=True, withMean=False)

# 5. FINAL VECTOR ASSEMBLER
assembler_inputs = [c+"_vec" for c in cat_cols] + ["scaled_num_features"]
final_assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features")

# 6. BUILD THE PIPELINE
stages = [sql_trans] + indexers + encoders + [num_assembler, scaler, final_assembler]
pipeline = Pipeline(stages=stages)

# 7. FIT PIPELINE STRICTLY ON TRAIN DATA (Rubric requirement)
print("Fitting the Pipeline strictly on Training data (preventing data leakage)...")
pipeline_model = pipeline.fit(train_df)

# 8. TRANSFORM ALL SPLITS
print("Transforming all splits...")
train_prepared = pipeline_model.transform(train_df)
val_prepared = pipeline_model.transform(val_df)
test_prepared = pipeline_model.transform(test_df)

# Show the resulting features column
print("\nSample of engineered features:")
train_prepared.select("features", "has_tip", "tip_amount").show(5, truncate=False)

# 9. SAVE THE FITTED PIPELINE TO DISK
pipeline_path = "../models/preprocessing_pipeline"
print(f"\nSaving the fitted pipeline to {pipeline_path}...")
pipeline_model.write().overwrite().save(pipeline_path)

print("Stage 6 completed successfully!")

26/05/27 02:15:03 WARN Utils: Your hostname, LaptopPablo resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/27 02:15:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/27 02:15:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Loading Train, Validation, and Test splits...


Fitting the Pipeline strictly on Training data (preventing data leakage)...



[Stage 9:=========>                                               (3 + 16) / 19]



Transforming all splits...



Sample of engineered features:


+----------------------------------------------------------------------------------------------------------------------------------------------------------+-------+----------+
|features                                                                                                                                                  |has_tip|tip_amount|
+----------------------------------------------------------------------------------------------------------------------------------------------------------+-------+----------+
|(40,[0,8,11,33,34,35,36,37,39],[1.0,1.0,1.0,0.7639242451191219,1.2968343976953038,1.9562094326317645,1.340829177055511,65.49956623548785,318.96880726747])|1      |2.25      |
|(40,[0,8,11,33,34,35,36,37,39],[1.0,1.0,1.0,0.7639242451191219,1.0498183219438173,1.8583989610001763,1.340829177055511,65.49956623548785,318.96880726747])|1      |2.15      |
|(40,[0,8,11,33,34,35,36,37,39],[1.0,1.0,1.0,0.7639242451191219,0.5557861704408446,1.2715361312106468,1.340829177055511,

Stage 6 completed successfully!
